# 02 — Sample Entity Lookup
Test partial and exact company name lookups.

In [8]:
import sys
sys.path.insert(0, "..")

In [9]:
import pandas as pd
from src.config import get_neo4j_settings
from src.storage.neo4j_repository import Neo4jRepository

settings = get_neo4j_settings()
repo = Neo4jRepository(**vars(settings))
repo.test_connection()
print("Connected")

Connected


In [10]:
# Verify the full-text index exists and is online before searching
index_rows = repo.run_query(
    "SHOW INDEXES YIELD name, state WHERE name = 'company_name_ft'"
)
if not index_rows:
    print("WARNING: company_name_ft index not found. Create it first:")
    print("  CREATE FULLTEXT INDEX company_name_ft IF NOT EXISTS")
    print("  FOR (n:Company) ON EACH [n.name];")
elif index_rows[0]["state"] != "ONLINE":
    print(f"WARNING: index state is {index_rows[0]['state']} — wait for ONLINE")
else:
    print("company_name_ft index: ONLINE")


company_name_ft index: ONLINE


## Partial match
Case-insensitive substring search. Edit `search_term` to explore.

In [11]:
search_term = "holdings"

results = repo.find_company_by_name(search_term, limit=10)
print(f"find_company_by_name({search_term!r}) → {len(results)} result(s)")

if results:
    import pandas as pd
    df = pd.DataFrame(results)
    df["score"] = df["score"].round(3)
    display(df[["name", "company_number", "status", "country_of_origin", "score"]])
else:
    print("No results — try a different search term.")


find_company_by_name('holdings') → 10 result(s)


,name,company_number,status,country_of_origin,score
0,HOLDINGS & HOLDINGS LTD,15829526,Active,United Kingdom,2.451
1,Arch Financial Holdings (Uk) Holdings,06410770,None,None,2.101
2,EJL HOLDINGS,10197080,Active,United Kingdom,2.075
3,FARTHING HOLDINGS,10598435,Active,United Kingdom,2.075
4,LUKELLE HOLDINGS,10818494,Active,United Kingdom,2.075
5,WRCQ HOLDINGS,11623634,Active,United Kingdom,2.075
6,BARNSTORMER HOLDINGS,10726655,Active,United Kingdom,2.075
7,SPEL HOLDINGS,08233395,Active,United Kingdom,2.075
8,LITTLESHAW HOLDINGS,13705032,Active,United Kingdom,2.075
9,OWLEM HOLDINGS,11277938,Active,United Kingdom,2.075


## Exact match
Case-sensitive equality on `Company.name`.

In [12]:
# Copy an exact name from the partial-match results above
exact_name = results[0]["name"] if results else "EXAMPLE HOLDINGS LTD"

company = repo.get_company_by_exact_name(exact_name)
print(f"get_company_by_exact_name({exact_name!r})")

if company:
    display(pd.DataFrame([company]))
else:
    print("Not found — check the name is an exact match.")

get_company_by_exact_name('HOLDINGS & HOLDINGS LTD')


,name,company_number,status,country_of_origin
0,HOLDINGS & HOLDINGS LTD,15829526,Active,United Kingdom


## Miss case — name not in graph

In [13]:
missing = repo.get_company_by_exact_name("THIS COMPANY DOES NOT EXIST LTD")
print(f"Result: {missing}")  # expected: None

Result: None


## Close

In [14]:
repo.close()
print("Driver closed")

Driver closed
